In [101]:
import tensorflow as tf
import numpy as np
import tensorflow.keras as keras
from mnist import datasets_url
from sklearn.model_selection import train_test_split
import unicodedata
import re
import os
import io


In [102]:
path_to_file = "C:/Users/LOQ/.keras/datasets//spa-eng.zip"

In [103]:
path_to_file=os.path.join(os.path.dirname(path_to_file),'spa-eng','spa.txt')

In [104]:
path_to_file

'C:/Users/LOQ/.keras/datasets\\spa-eng\\spa.txt'

In [105]:
def unicode_to_ascii(s):
    return''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c)!='MN')

In [106]:
def preprocess_sentence(w):
    w=unicode_to_ascii(w.lower().strip())
    w=re.sub(r"([.!?~,])", r" \1", w)
    w=re.sub(r'([" "])+', " ", w)
    w=re.sub(r"([^a-zA-Z?.~])+", " ", w)
    w=w.rstrip().strip()
    w='<start>'+w+'<end>'
    return w

In [107]:
en_sentence="I have some friends to help."
preprocess_sentence(en_sentence)

'<start>i have some friends to help .<end>'

In [108]:
en_sentence2="Párate aquí"
preprocess_sentence(en_sentence2)

'<start>pa rate aqui<end>'

In [109]:
def create_dataset(path,num_examples):
    lines=io.open(path,encoding='utf-8').read().strip().split('\n')
    word_paris= [[preprocess_sentence(w) for w in l.split('\t' )]for l in lines[:num_examples]]
    print(list(zip(word_paris)))
    return zip(*word_paris)

In [110]:
print(create_dataset(path_to_file,2))

[(['<start>go .<end>', '<start>ve .<end>'],), (['<start>go .<end>', '<start>vete .<end>'],)]


In [111]:
en,sp=create_dataset(path_to_file,None)


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [112]:
len(en)

118964

In [113]:
len(sp)

118964

In [114]:
def max_lenght(tensor):
    return max(len(t) for t in tensor)

In [115]:
def tokenaze(lang):
    lang_tokenizer=keras.preprocessing.text.Tokenizer(filters='')
    lang_tokenizer.fit_on_texts(lang)
    tensor=lang_tokenizer.texts_to_sequences(lang)
    tensor=keras.preprocessing.sequence.pad_sequences(tensor,padding='post')
    return tensor,lang_tokenizer

In [116]:
def load_dataset(path,num_examples):
    targ_lang,inp_lang=create_dataset(path, num_examples)
    input_tensor,inp_lang_tokenizer=tokenaze(inp_lang)
    target_tensor,target_lang_tokenizer=tokenaze(targ_lang)
    return input_tensor,target_tensor,inp_lang_tokenizer,target_lang_tokenizer

In [117]:
input_tensor,target_tensor,inp_lang_tokenizer,target_lang_tokenizer=load_dataset(path_to_file,20000)

[(['<start>go .<end>', '<start>ve .<end>'],), (['<start>go .<end>', '<start>vete .<end>'],), (['<start>go .<end>', '<start>vaya .<end>'],), (['<start>go .<end>', '<start>va yase .<end>'],), (['<start>hi .<end>', '<start>hola .<end>'],), (['<start>run<end>', '<start>corre<end>'],), (['<start>run .<end>', '<start>corred .<end>'],), (['<start>who ?<end>', '<start>quie n ?<end>'],), (['<start>fire<end>', '<start>fuego<end>'],), (['<start>fire<end>', '<start>incendio<end>'],), (['<start>fire<end>', '<start>disparad<end>'],), (['<start>help<end>', '<start>ayuda<end>'],), (['<start>help<end>', '<start>socorro auxilio<end>'],), (['<start>help<end>', '<start>auxilio<end>'],), (['<start>jump<end>', '<start>salta<end>'],), (['<start>jump .<end>', '<start>salte .<end>'],), (['<start>stop<end>', '<start>parad<end>'],), (['<start>stop<end>', '<start>para<end>'],), (['<start>stop<end>', '<start>pare<end>'],), (['<start>wait<end>', '<start>espera<end>'],), (['<start>wait .<end>', '<start>esperen .<end

In [118]:
len(input_tensor)

20000

In [119]:
input_tensor

array([[206,   1,   0, ...,   0,   0,   0],
       [239,   1,   0, ...,   0,   0,   0],
       [776,   1,   0, ...,   0,   0,   0],
       ...,
       [  6,  28,  34, ...,   0,   0,   0],
       [  6,  28,  34, ...,   0,   0,   0],
       [  6,  28,  34, ...,   0,   0,   0]],
      shape=(20000, 15), dtype=int32)

In [120]:
max_leng_targ,max_leng_inp=max_lenght(target_tensor),max_lenght(input_tensor)

In [121]:
x_train,x_test,y_train,y_test=train_test_split(input_tensor,target_tensor,test_size=0.2)

In [122]:
def convert(lang,tensor):
    for t in tensor:
        if t !=0:
            print(t,' .... ',lang.index_word[t])

In [123]:
convert(inp_lang_tokenizer,input_tensor[0])

206  ....  <start>ve
1  ....  .<end>


In [124]:
input_tensor[0]

array([206,   1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0], dtype=int32)

In [125]:
BUFFER_SIZE=len(x_train)
BATCH_SIZE=64
steps_pre_epoch=len(x_train)//BATCH_SIZE
embedding_dim=256
units=1024
vocab_inp_size=len(inp_lang_tokenizer.word_index)+1
vocab_target_size=len(target_lang_tokenizer.word_index)+1


In [126]:
dataset=tf.data.Dataset.from_tensor_slices((x_train,y_train)).shuffle(BUFFER_SIZE)
dataset=dataset.batch(BATCH_SIZE,drop_remainder=True)

In [127]:
class Encoder(keras.Model):
    def __init__(self,vocab_size,embedding_dim,encoder_units,batch_size):
        super(Encoder, self).__init__()
        self.encoder_units=encoder_units
        self.batch_size=batch_size
        self.embedding=keras.layers.Embedding(vocab_size,embedding_dim)
        self.gru=keras.layers.GRU(self.encoder_units,return_sequences=True,return_state=True)

    def call(self,x,hidden):
        x=self.embedding(x)
        output,state=self.gru(x,initial_state=hidden)
        return output,state
    def initiillize_hidden_state(self):
        return tf.zeros((self.batch_size,self.encoder_units))

In [128]:
encoder=Encoder(vocab_inp_size,embedding_dim,units,BATCH_SIZE)

In [129]:
encoder

<Encoder name=encoder_1, built=False>

In [130]:
simple_hidden=encoder.initiillize_hidden_state()
simple_hidden

<tf.Tensor: shape=(64, 1024), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(64, 1024), dtype=float32)>

In [131]:
dataset

<_BatchDataset element_spec=(TensorSpec(shape=(64, 15), dtype=tf.int32, name=None), TensorSpec(shape=(64, 7), dtype=tf.int32, name=None))>

In [132]:
example_input_batch,example_target_batch=next(iter(dataset))

In [142]:
simple_output,simple_states=encoder(example_input_batch,simple_hidden)

In [143]:
class Attention(keras.layers.Layer):
    def __init__(self,units):
        super(Attention,self).__init__()
        self.w1=keras.layers.Dense(units)
        self.w2=keras.layers.Dense(units)
        self.V=keras.layers.Dense(1)
    def call(self,query,values):
        hidden_with_time_axis=tf.expand_dims(query,1)
        score=self.V(tf.nn.tanh(self.w1(values)+self.w2(hidden_with_time_axis)))
        attention_weights=tf.nn.softmax(score,axis=1)
        convert_vector=attention_weights*values
        convert_vector=tf.reduce_sum(convert_vector,axis=1)
        return convert_vector,attention_weights

In [144]:
attention_layer=Attention(10)
attention_layer(simple_hidden,simple_output)

(<tf.Tensor: shape=(64, 1024), dtype=float32, numpy=
 array([[ 0.0130413 , -0.00559005, -0.00714784, ..., -0.01305268,
          0.00100331,  0.01177196],
        [ 0.0147219 , -0.0060457 , -0.01101038, ..., -0.01044722,
          0.00480841,  0.01407806],
        [ 0.00713834, -0.00473972, -0.00299719, ..., -0.00490984,
          0.00109936,  0.0083791 ],
        ...,
        [ 0.01390926, -0.00690792, -0.01054217, ..., -0.01031058,
          0.00349969,  0.01231663],
        [ 0.01551071, -0.00575881, -0.00973307, ..., -0.01197149,
          0.00285254,  0.01264881],
        [ 0.01288389, -0.00653862, -0.0105438 , ..., -0.01313369,
          0.00359657,  0.01320377]], shape=(64, 1024), dtype=float32)>,
 <tf.Tensor: shape=(64, 15, 1), dtype=float32, numpy=
 array([[[0.06541064],
         [0.06747754],
         [0.06693487],
         [0.06549541],
         [0.06612971],
         [0.06649019],
         [0.06669276],
         [0.06680644],
         [0.06687125],
         [0.06690955],
  

In [146]:
attention_result,attention_weights=attention_layer(simple_hidden,simple_output)

In [147]:
class Decoder(keras.Model):
    def __init__(self,vocab_size,embedding_dim,units,batch_size):
        super(Decoder, self).__init__()
        self.batch_size=batch_size
        self.dec_units=dec_units
        self.embedding=keras.layers.Embedding(vocab_size,embedding_dim)
        self.gru=keras.layers.GRU(self.dec_units,return_sequences=True,return_state=True)
        self.fc=keras.layers.Dense(vocab_size)
        self.attention=Attention(self.dec_units)